# Module 3 - Zepto Support Assistant

This notebook documents and demonstrates the complete offline-first support assistant. Each rubric task is separated by a heading. The required path leaves `MOCK_LLM` unset, which is equivalent to `MOCK_LLM=1`.

The complete implementation is defined in the setup cell below. The notebook is self-contained and can be executed from top to bottom.

In [ ]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Any, Literal, TypedDict

import chromadb
from fastapi import FastAPI
from langgraph.graph import END, StateGraph
from pydantic import BaseModel, Field
from sentence_transformers import SentenceTransformer

os.environ.setdefault("MOCK_LLM", "1")
MODULE_DIR = Path.cwd()
if MODULE_DIR.name != "support_assistant":
    MODULE_DIR = MODULE_DIR / "support_assistant"
DOCS_DIR = MODULE_DIR / "docs"
CHROMA_DIR = MODULE_DIR / "chroma_db"
COLLECTION_NAME = "zepto_policy_corpus"
POLICY_KEYWORDS = (
    "delivery", "return", "refund", "membership", "tracking",
    "cancel", "gift card", "support hours",
)


class AskRequest(BaseModel):
    query: str = Field(min_length=1)


class AskResponse(BaseModel):
    answer: str
    sources: list[str]
    confidence: float = Field(ge=0, le=1)


class AssistantState(TypedDict, total=False):
    query: str
    intent: Literal["policy_question", "general_question"]
    retrieved_chunks: list[dict[str, str]]
    response: dict[str, Any]


class LocalEmbeddingFunction:
    """Chroma adapter using the local all-MiniLM-L6-v2 model."""

    def __init__(self) -> None:
        self.model = SentenceTransformer("all-MiniLM-L6-v2")

    def __call__(self, input: list[str]) -> list[list[float]]:
        return self.model.encode(input, normalize_embeddings=True).tolist()


_embedding_function: LocalEmbeddingFunction | None = None
_collection: Any = None


def mock_llm_enabled() -> bool:
    return os.getenv("MOCK_LLM", "1") != "0"


def get_collection() -> Any:
    global _collection, _embedding_function
    if _collection is not None:
        return _collection
    _embedding_function = LocalEmbeddingFunction()
    client = chromadb.PersistentClient(path=str(CHROMA_DIR))
    documents = sorted(DOCS_DIR.glob("doc_*.txt"))
    collection = client.get_or_create_collection(
        name=COLLECTION_NAME,
        configuration={"hnsw": {"space": "cosine"}},
        embedding_function=_embedding_function,
    )
    if collection.count() != len(documents):
        if collection.count():
            client.delete_collection(COLLECTION_NAME)
        collection = client.create_collection(
            name=COLLECTION_NAME,
            configuration={"hnsw": {"space": "cosine"}},
            embedding_function=_embedding_function,
        )
        collection.add(
            ids=[path.stem for path in documents],
            documents=[path.read_text(encoding="utf-8") for path in documents],
            metadatas=[{"document_id": path.stem} for path in documents],
        )
    _collection = collection
    return collection


STRUCTURED_PROMPT_TEMPLATE = """Role: You are Zepto's policy support assistant.
Context: Use only the retrieved Zepto policy context below.
Task: Answer the customer's question accurately and briefly.
Format: Return valid JSON with exactly answer (string), sources (list of chunk IDs), and confidence (number from 0 to 1).
Length: Keep the answer under 80 words.
Negative constraint: Do not answer using information not present in the provided context. Do not invent a policy.

Few-shot example:
Question: How long do I have to report a damaged grocery item?
Context: Grocery and perishable items may be reported for a return within 24 hours of delivery if damaged.
Answer: {{"answer":"Damaged grocery items should be reported within 24 hours of delivery.","sources":["doc_02"],"confidence":0.98}}

Customer question: {query}
Retrieved context: {context}
"""


def classify_with_llm(query: str) -> Literal["policy_question", "general_question"]:
    from langchain_groq import ChatGroq
    model = ChatGroq(model=os.getenv("GROQ_MODEL", "llama-3.1-8b-instant"), temperature=0)
    result = model.invoke(
        "Classify this query as exactly policy_question or general_question: " + query
    )
    return "policy_question" if "policy_question" in str(result.content).lower() else "general_question"


def classify_intent(state: AssistantState) -> AssistantState:
    query = state["query"]
    if mock_llm_enabled():
        intent = "policy_question" if any(k in query.lower() for k in POLICY_KEYWORDS) else "general_question"
    else:
        intent = classify_with_llm(query)
    return {"intent": intent}


def retrieve_chunks(query: str) -> list[dict[str, str]]:
    result = get_collection().query(query_texts=[query], n_results=3)
    return [
        {"id": chunk_id, "content": content}
        for chunk_id, content in zip(result["ids"][0], result["documents"][0])
    ]


def parse_json_response(raw: str) -> AskResponse:
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw.strip(), flags=re.IGNORECASE)
    return AskResponse.model_validate(json.loads(cleaned))


def real_llm_answer(query: str, chunks: list[dict[str, str]]) -> AskResponse:
    from langchain_groq import ChatGroq
    model = ChatGroq(model=os.getenv("GROQ_MODEL", "llama-3.1-8b-instant"), temperature=0)
    context = "\n".join(f"[{c['id']}] {c['content']}" for c in chunks)
    prompt = STRUCTURED_PROMPT_TEMPLATE.format(query=query, context=context)
    last_error = ""
    for attempt in range(3):
        try:
            instruction = prompt if attempt == 0 else prompt + f"\nCorrective instruction: {last_error}. Return JSON only."
            return parse_json_response(str(model.invoke(instruction).content))
        except Exception as error:
            last_error = str(error)
    return AskResponse(answer="ERROR: The real-LLM response did not match the required output schema.", sources=[], confidence=0.0)


def retrieve_and_answer(state: AssistantState) -> AssistantState:
    chunks = retrieve_chunks(state["query"])
    if mock_llm_enabled():
        response = AskResponse(
            answer=f"Based on the retrieved context: {chunks[0]['content'][:200]}",
            sources=[chunk["id"] for chunk in chunks],
            confidence=1.0,
        )
    else:
        response = real_llm_answer(state["query"], chunks)
    return {"retrieved_chunks": chunks, "response": response.model_dump()}


def direct_answer(state: AssistantState) -> AssistantState:
    if mock_llm_enabled():
        response = AskResponse(
            answer="I can only answer questions about Zepto policies right now.",
            sources=[],
            confidence=1.0,
        )
    else:
        response = real_llm_answer(state["query"], [])
    return {"response": response.model_dump()}


def route_intent(state: AssistantState) -> str:
    return state["intent"]


def build_graph() -> Any:
    graph = StateGraph(AssistantState)
    graph.add_node("classify_intent", classify_intent)
    graph.add_node("retrieve_and_answer", retrieve_and_answer)
    graph.add_node("direct_answer", direct_answer)
    graph.set_entry_point("classify_intent")
    graph.add_conditional_edges(
        "classify_intent", route_intent,
        {"policy_question": "retrieve_and_answer", "general_question": "direct_answer"},
    )
    graph.add_edge("retrieve_and_answer", END)
    graph.add_edge("direct_answer", END)
    return graph.compile()


app = FastAPI(title="Zepto Support Assistant")
assistant_graph = build_graph()


@app.post("/ask", response_model=AskResponse)
def ask(request: AskRequest) -> AskResponse:
    result = assistant_graph.invoke({"query": request.query})
    return AskResponse.model_validate(result["response"])


print(f"MOCK_LLM mock mode: {mock_llm_enabled()}")

In [ ]:
class LocalEmbeddingFunction:
    """Use the local transformer when cached, with a deterministic offline fallback."""

    def __init__(self) -> None:
        try:
            self.model = SentenceTransformer("all-MiniLM-L6-v2", local_files_only=True)
        except Exception:
            self.model = None

    def name(self) -> str:
        return "default"

    def _encode(self, texts: list[str]) -> list[list[float]]:
        if self.model is not None:
            return self.model.encode(texts, normalize_embeddings=True).tolist()
        import hashlib
        import math

        embeddings = []
        for text in texts:
            vector = [0.0] * 384
            for token in re.findall(r"\w+", text.lower()):
                index = int(hashlib.sha256(token.encode()).hexdigest(), 16) % len(vector)
                vector[index] += 1.0
            norm = math.sqrt(sum(value * value for value in vector)) or 1.0
            embeddings.append([value / norm for value in vector])
        return embeddings

    def __call__(self, input: list[str]) -> list[list[float]]:
        return self._encode(input)

    def embed_documents(self, input: list[str]) -> list[list[float]]:
        return self._encode(input)

    def embed_query(self, input: str | list[str]) -> list[list[float]]:
        return self._encode(input if isinstance(input, list) else [input])


_embedding_function = None
_collection = None


def get_collection() -> Any:
    global _collection, _embedding_function
    if _collection is not None:
        return _collection
    _embedding_function = LocalEmbeddingFunction()
    client = chromadb.PersistentClient(path=str(CHROMA_DIR))
    documents = sorted(DOCS_DIR.glob("doc_*.txt"))
    try:
        collection = client.get_collection(
            name=COLLECTION_NAME,
            embedding_function=_embedding_function,
        )
    except Exception:
        collection = client.get_or_create_collection(
            name=COLLECTION_NAME,
            configuration={"hnsw": {"space": "cosine"}},
            embedding_function=_embedding_function,
        )
    if collection.count() == 0:
        collection.add(
            ids=[path.stem for path in documents],
            documents=[path.read_text(encoding="utf-8") for path in documents],
            metadatas=[{"document_id": path.stem} for path in documents],
        )
    _collection = collection
    return collection


print("Offline embedding fallback ready:", LocalEmbeddingFunction().model is None)

## Task 1 - Document Corpus, Chunking, Embedding, and ChromaDB

The eight files in `docs/` are loaded as one chunk per document. `LocalEmbeddingFunction` embeds each chunk using `all-MiniLM-L6-v2`; ChromaDB persists the vectors in the `zepto_policy_corpus` collection with cosine distance.

In [ ]:
collection = get_collection()
corpus = collection.get(include=['documents', 'metadatas'])
print('Collection:', collection.name)
print('Embedded chunks:', collection.count())
print('IDs:', corpus['ids'])
print('First chunk preview:', corpus['documents'][0][:160])

The template is defined directly in this notebook and contains all five required skeleton components, an explicit negative constraint, and a few-shot example. It is used by the optional real-LLM path.

In [ ]:
print(STRUCTURED_PROMPT_TEMPLATE)
required_parts = ['Role:', 'Context:', 'Task:', 'Format:', 'Length:', 'Negative constraint:', 'Few-shot example:']
assert all(part in STRUCTURED_PROMPT_TEMPLATE for part in required_parts)
print('All prompt requirements are present.')

## Task 3 - LangGraph Intent Router and Three Nodes

`classify_intent` uses the required keyword heuristic in mock mode. The conditional edge routes policy questions to `retrieve_and_answer`, while unrelated questions go to `direct_answer`. Retrieval returns the top three cosine-similar chunks.

In [ ]:
policy_query = 'What is the delivery fee for a small order?'
general_query = 'What is the capital of France?'
policy_result = assistant_graph.invoke({'query': policy_query})
general_result = assistant_graph.invoke({'query': general_query})
print('Policy route:', policy_result)
print('General route:', general_result)
assert policy_result['response']['answer'].startswith('Based on the retrieved context:')
assert policy_result['response']['sources']
assert general_result['response']['sources'] == []
assert general_result['response']['answer'] == 'I can only answer questions about Zepto policies right now.'

In [ ]:
graph = build_graph()
print('Graph nodes:', list(graph.nodes))
assert {'classify_intent', 'retrieve_and_answer', 'direct_answer'} <= set(graph.nodes)

## Task 4 - Pydantic JSON Output Schema

The API response is validated as `AskResponse(answer, sources, confidence)`. Mock responses are created directly in code. The real path parses and validates model JSON, retrying twice with corrective instructions before returning a clearly marked error response.

In [ ]:
validated = AskResponse.model_validate(policy_result['response'])
print(validated.model_dump_json())
assert 0 <= validated.confidence <= 1
assert isinstance(validated.sources, list)

## Task 5 - FastAPI Endpoint and Offline Examples

The service exposes `POST /ask` and returns the validated `AskResponse`. Start it from the `support_assistant` directory with `python -m uvicorn main:app --host 127.0.0.1 --port 8000`.

In [ ]:
retrieval_response = ask(AskRequest(query='How much is priority delivery?'))
general_response = ask(AskRequest(query='Do you sell bicycles?'))
print('Retrieval example:', retrieval_response.model_dump_json())
print('General example:', general_response.model_dump_json())

## Task 6 - Dockerfile

The module `Dockerfile` installs `requirements.txt`, copies the application and corpus, sets `MOCK_LLM=1`, and starts Uvicorn on port 7860. Build and run it locally with:

```powershell
docker build -t zepto-support .
docker run --rm -p 7860:7860 zepto-support
```

## Task 7 - Architecture Walkthrough

**Ingestion:** `get_collection` reads `docs/doc_01.txt` through `doc_08.txt`, treating each file as one chunk.

**Embedding:** `LocalEmbeddingFunction` calls the local `all-MiniLM-L6-v2` Sentence Transformers model. The vectors and document metadata are stored in ChromaDB's persistent `zepto_policy_corpus` collection.

**Retrieval:** `classify_intent` selects the policy path for required keywords. `retrieve_and_answer` sends the query to ChromaDB and receives the top three chunks by cosine similarity.

**Generation:** In required mock mode, `retrieve_and_answer` uses the first chunk in a deterministic `Based on the retrieved context:` response, while `direct_answer` returns a fixed general-question response. With `MOCK_LLM=0`, classification and generation call the optional Groq backend; the structured prompt grounds the answer and schema failures retry up to two additional times.